In [ ]:
# PrandialTwin on Kaggle: three more forecasting baselines (Moirai, MOMENT, TS2Vec), one model per run, on all three
# datasets: CGMacros (5 folds), ShanghaiT2DM (ext) and the six simulated Indian regions.
# Settings: Accelerator = GPU T4 x2 (one is used), Internet = On. Add Input: your PRIVATE dataset holding prandialtwin_kit.zip.
# Then "Save Version" -> "Save & Run All (Commit)". Download out_<model>/ from Output (small: a few MB).
MODEL = "moirai"                              # moirai | moment | ts2vec: one per account, so they run at the same time
PARTS = ("fold0 fold1 fold2 fold3 fold4 ext "
         "india_north india_south india_east india_west india_central india_northeast")

import glob, os, shutil, subprocess
src = os.path.dirname(glob.glob("/kaggle/input/**/tsfm/tsfm_run.py", recursive=True)[0])
work = "/kaggle/working/tsfm"
shutil.copytree(src, work, dirs_exist_ok=True)  # inputs are read-only
# Each model's library without its old pins (torch < 2.5, numpy 1.x), plus exactly what its code imports.
# Versions are the ones tested locally (torch 2.14, numpy 2.2).
setup = {"moirai": "pip install -q --no-deps uni2ts==2.0.0 && pip install -q einops jaxtyping==0.2.38 gluonts==0.17.0 lightning==2.6.6 "
                   "hydra-core==1.3.7 python-dotenv && (python -c 'import jax' || pip install -q 'jax[cpu]')",
         "moment": "pip install -q --no-deps momentfm==0.1.4 && pip install -q transformers==5.20.1",
         "ts2vec": "git clone -q https://github.com/zhihanyue/ts2vec && git -C ts2vec checkout -q b0088e14a99706c05451316dc6db8d3da9351163"}
subprocess.run(setup[MODEL], shell=True, check=True, cwd=work)

try:  # each part's forecasts are written as soon as it is done, so a timeout loses only the part that was running
    run = subprocess.Popen(f"PYTHONWARNINGS=ignore python -u tsfm_run.py {MODEL} {PARTS}", shell=True, cwd=work,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in run.stdout:  # streamed into this cell / the Log tab
        print(line, end="", flush=True)
    assert run.wait() == 0, "failed: see the lines above"
finally:
    if os.path.isdir(f"{work}/out"):
        shutil.copytree(f"{work}/out", f"/kaggle/working/out_{MODEL}", dirs_exist_ok=True)
        print("output:", sorted(os.listdir(f"/kaggle/working/out_{MODEL}")))
    shutil.rmtree(work)